# Chapter 4 Lab — Minimal RAG Pipeline (embed → search → inject)

**Estimated time:** 25 minutes · **Read-and-run.** A toy hashing embedder stands in for a
real model such as **NVIDIA NeMo Retriever**; the pipeline shape is identical.

In [ ]:
import math, hashlib
DOCS = {
 'd1':'LoRA adds low-rank adapters so you fine-tune a fraction of the weights.',
 'd2':'RAG injects retrieved context into the prompt; good when knowledge changes often.',
 'd3':'Full fine-tuning updates all weights and needs the most labeled data and compute.',
 'd4':'Triton Inference Server handles dynamic batching for high-throughput serving.'}

DIM=64
def embed(text):
    v=[0.0]*DIM
    for tok in text.lower().split():
        h=int(hashlib.md5(tok.encode()).hexdigest(),16)
        v[h%DIM]+= 1.0
    n=math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/n for x in v]
def cosine(a,b): return sum(x*y for x,y in zip(a,b))

In [ ]:
index = {k: embed(t) for k,t in DOCS.items()}
def retrieve(query, k=2):
    q=embed(query)
    scored=sorted(((cosine(q,v),did) for did,v in index.items()), reverse=True)
    return [(did, DOCS[did], round(s,3)) for s,did in scored[:k]]

query='When should I use RAG instead of fine-tuning?'
hits=retrieve(query)
for did,txt,s in hits: print(f'{did} (sim={s}): {txt}')

## Build the augmented prompt

In [ ]:
context='\n'.join(f'- {txt}' for _,txt,_ in hits)
prompt=f'''Use ONLY the context to answer.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'''
print(prompt)

## Key takeaway
The prompt-vs-RAG-vs-fine-tune decision hinges on **how often knowledge changes**, **how much
labeled data exists**, and the **latency/cost budget**. RAG wins when facts change frequently
and you cannot retrain on every change.